QLoRA, an efficient finetuning method that lets you finetune very large pretrained language models (e.g., 65B parameters) on a single 48GB GPU by backpropagating only into LoRA adapters while keeping the base model frozen and quantized to 4 bits. It highlights three key engineering innovations—NF4 (4‑bit NormalFloat), Double Quantization, and Paged Optimizers—that reduce memory use without hurting full‑precision finetuning performance, and reports strong empirical results (the Guanaco family, Vicuna benchmark, GPT‑4 evaluations, etc.).

QLoRA’s idea: QLoRA keeps that same LoRA training paradigm (train only the low‑rank adapters) but quantizes the frozen base model to 4 bits and still backpropagates gradients through the frozen, quantized model into the LoRA adapters. In other words, the `base weights` are stored and used in `4‑bit form` (to save memory), but gradient flow is routed so the adapters receive correct updates even though the main model is frozen and quantized.

![Different finetuning methods and their memory requirements](image-2.png)

**Full Finetuning**: Base model stored in 16‑bit. All weights updated; optimizer state is large (32‑bit or 16‑bit copies), so GPU memory usage is high. Blue arrows indicate many parameter updates; green arrows show gradients flowing through every parameter.

**LoRA**: Base model frozen and kept in 16‑bit. Small low‑rank adapter matrices (16‑bit) are added to transformer layers and are the only trainable parameters. Gradients flow through the base model to the adapters, but optimizer state is only for adapters, so memory is much lower than full finetuning.

**QLoRA**: Base model quantized to 4‑bit and kept frozen. LoRA adapters remain trainable. Paged optimizer and CPU paging are used to move large optimizer state off GPU when needed. The diagram shows pink arrows for paging flow (GPU ↔ CPU), green arrows for gradient flow into adapters, and blue arrows for adapter parameter updates.

gradient flow: gradients computed during backpropagation travel through the frozen base model into the adapters so the adapters receive learning signals.

parameter updates: optimizer updates applied only to trainable parameters (all parameters in full finetuning; only adapters in LoRA/QLoRA).

paging flow: memory movement between GPU and CPU to avoid GPU memory spikes; present only in QLoRA to handle optimizer state and temporary tensors.

QLoRA = LoRA + aggressive 4‑bit quantization + paging; maximizes memory efficiency while preserving finetuning quality.

## block‑wise k‑bit (i.e. 2 bit, 4 bit , etc ...) quantization:

Block‑wise k‑bit quantization splits a large tensor into smaller contiguous blocks and quantizes each block independently to 𝑘 bits using its own scale (quantization constant). This `avoids a single outlier value` from dominating the scale for the whole tensor and preserves more information in typical (non‑outlier) values.

***Qus: Where this Outliear word come from and how It releated with k‑bit (i.e. 2 bit, 4 bit , etc ...) quantization ?***

TODO

#### Math behind on uniform symmetric quantization (per block):

***Qus: In QLORA why uniform symmetric quantization (per block). It's necessary conditions for quantization or there are possibility for non-uniform quantization ?***

TODO

***Qus: In QLoRA mentions word block‑wise k‑bit? What is meaning of block?***

If you quantize the entire tensor with a single scale and there is one huge outlier, the global max∣𝑥∣ becomes very large and most other values collapse toward zero after quantization. By splitting into blocks, each block’s scale reflects its local dynamic range, so non‑outlier blocks keep more representational fidelity.

### Symmetric $k$-bit Quantization (Absmax Technique)

Let a block of real weights be $B = [x_1, \dots, x_m]$ with $x_j \in \mathbb{R}$. For symmetric $k$-bit quantization (signed), the integer range is defined by:

$$I = \{-Q, \dots, -1, 0, 1, \dots, Q\}, \qquad Q = 2^{k-1} - 1$$

#### Step 1 — Compute block scale (absmax)

Find the absolute maximum value in the block to determine the scaling factor $s$:

$$s = \frac{Q}{\max_{j} |x_j|}$$

step size = $$\qquad \Delta = \frac{1}{s} = \frac{\max|x|}{Q}.$$

A simple model for the quantization noise variance (assuming uniform error in a step) is

$$\text{Var}(\text{error}) \approx \frac{\Delta^2}{12} = \frac{\max|x|^2}{12 Q^2}.$$

### Error estimate (simple bound)

For symmetric uniform quantization on a block, the maximum absolute quantization error per element is:

$$|x_j - \hat{x}_j| \leq \frac{1}{2s} = \frac{\max_{j}|x_j|}{2Q}$$

So increasing $k$ (larger $Q$) or reducing block max reduces worst-case error.

**Key dependence:** error $\propto \frac{\max|x|^2}{Q^2}$ and $Q \approx 2^{k-1}$, so error scales roughly like $2^{-2k}$ for fixed block max.

#### Step 2 — Quantize each element (rounding to integer)

Scale the real numbers, round them to the nearest integer, and clip them to ensure they stay within the valid range $[-Q, Q]$:

$$x_j^{\text{int}} = \text{clip}\left(\text{round}(s \cdot x_j), -Q, Q\right)$$

#### Step 3 — Dequantize (recover approximate float)

Reconstruct the approximate original floating-point values from the quantized integers by dividing by the scale:

$$\hat{x}_j = \frac{x_j^{\text{int}}}{s} = \frac{x_j^{\text{int}}}{Q} \cdot \max_{j} |x_j|$$


### Example 1 (block size $m = 4$, $k = 2$ bits)

Take a block $B = [0.12, -0.05, 0.9, 0.02]$. With $k = 2$ bits, the signed integer range is $\{-1, 0, 1\}$, so:

$$Q = 2^{2-1} - 1 = 1.$$


#### 1. Compute block absmax:
$$\max_{j} |x_j| = 0.9$$

#### 2. Scale:
$$s = \frac{Q}{\max_{j} |x_j|} = \frac{1}{0.9} \approx 1.1111$$

#### 3. Quantize each element:
* $0.12 \times s \approx 0.1333 \implies \text{round}(0.1333) = 0 \implies x_1^{\text{int}} = 0.$
* $-0.05 \times s \approx -0.0556 \implies \text{round}(-0.0556) = 0 \implies x_2^{\text{int}} = 0.$
* $0.9 \times s \approx 1.0 \implies \text{round}(1.0) = 1 \implies x_3^{\text{int}} = 1.$
* $0.02 \times s \approx 0.0222 \implies \text{round}(0.0222) = 0 \implies x_4^{\text{int}} = 0.$

So the quantized block (integers) is **$[0, 0, 1, 0]$**.

#### 4. Dequantize to floats:
* $\hat{x}_1 = \frac{0}{s} = 0.$
* $\hat{x}_2 = \frac{0}{s} = 0.$
* $\hat{x}_3 = \frac{1}{s} \approx 0.9.$
* $\hat{x}_4 = \frac{0}{s} = 0.$

Dequantized approximate block is **$[0., 0., 0.9, 0.]$**.

The large value $0.9$ is preserved (mapped to 1 then back to 0.9), while small values are rounded to zero. With only 2 bits this is expected; block‑wise quantization ensured the block’s dynamic range was set by 0.9 rather than by a single outlier elsewhere in the full tensor.

## Example 2 Block‑wise 4‑bit quantization:

quantize a small block of real numbers to signed 4‑bit integers (block‑wise symmetric uniform quantization), then dequantize and measure error.

### Example block ($m = 8$)

Let the block of weights be:

$$B = [0.12, -0.05, 0.90, 0.02, -0.80, 0.33, 0.01, -0.02]$$

#### 1. Block max absolute value
$$\max_{j}|x_j| = 0.90$$

#### 2. Scale
Using $Q = 7$ (which corresponds to 4-bit signed quantization where $Q = 2^{4-1} - 1 = 7$):

$$s = \frac{7}{0.90} \approx 7.7778$$

#### 3. Quantize (compute $s \cdot x_j$, round, clip)
* $0.12 \times s \approx 0.9333 \implies \text{round} = 1$
* $-0.05 \times s \approx -0.3889 \implies \text{round} = 0$
* $0.90 \times s \approx 7.0000 \implies \text{round} = 7$
* $0.02 \times s \approx 0.1556 \implies \text{round} = 0$
* $-0.80 \times s \approx -6.2222 \implies \text{round} = -6$
* $0.33 \times s \approx 2.5667 \implies \text{round} = 3$
* $0.01 \times s \approx 0.0778 \implies \text{round} = 0$
* $-0.02 \times s \approx -0.1556 \implies \text{round} = 0$

The final quantized integer array is **$[1, 0, 7, 0, -6, 3, 0, 0]$**.

### Quantized integers:

$$X^{\text{int}} = [1, 0, 7, 0, -6, 3, 0, 0]$$

#### 4. Dequantize

$$\hat{B} = \frac{X^{\text{int}}}{s} \approx [0.1286, 0, 0.9000, 0, -0.7714, 0.3857, 0, 0]$$

#### 5. Errors and summary

* **Elementwise errors** $e_j = x_j - \hat{x}_j$:

$$[-0.0086, -0.05, 0, 0.02, -0.0286, -0.0557, 0.01, -0.02]$$

* **Squared errors sum** $\approx 0.0073$
* **Mean squared error (MSE):**

$$\text{MSE} \approx \frac{0.0073}{8} \approx 0.0009125$$

* **Root MSE (RMSE):** $\approx 0.0302$

* **Worst-case theoretical bound for symmetric uniform quantization:**

$$|x_j - \hat{x}_j| \leq \frac{\max_{j}|x_j|}{2Q} = \frac{0.90}{2 \cdot 7} \approx 0.06429,$$

and the observed maximum absolute error $\approx 0.0557$ is below this bound.

Q. for two bit quantization you have taken block size 4, for 4 bit quantization  you have taken block size 8. Is their any realitions between block size nad k bit quantization?

There is no fixed rule that ties block size to 𝑘 (number of bits); instead they trade off against each other.
- More bits (larger 𝑘) reduce quantization step size and error exponentially.
- Smaller blocks reduce per‑block dynamic range (smaller local max), so they also reduce quantization error, but they increase metadata and CPU/GPU overhead.

### How block size interacts with $k$

* **Smaller block size** $\rightarrow$ smaller per-block $\max|x|$ in many tensors (because outliers are less likely to appear in every block) $\rightarrow$ **smaller $\Delta$** $\rightarrow$ lower error for the same $k$.
* **Larger block size** $\rightarrow$ higher chance a block contains an outlier $\rightarrow$ larger $\max|x|$ $\rightarrow$ **larger $\Delta$** $\rightarrow$ worse fidelity for same $k$.
* **But smaller blocks** mean **more blocks** $\rightarrow$ more per-block metadata (a scale per block) and more bookkeeping/paging overhead.

Quantitatively, if the global max is $M_{\text{global}}$ and a typical block max is $M_{\text{block}} \leq M_{\text{global}}$, then for fixed $k$ the variance ratio is

$$\frac{\text{Var}_{\text{block}}}{\text{Var}_{\text{global}}} \approx \left(\frac{M_{\text{block}}}{M_{\text{global}}}\right)^2.$$

### Numeric comparison (same block max, different $k$)

Take $\max|x| = 0.90$.

* **For $k = 2$:** $Q = 1, \Delta = 0.90$
    * Worst-case error $\leq 0.45$
    * Variance $\approx \frac{0.90^2}{12 \cdot 1^2} = 0.0675$

* **For $k = 4$:** $Q = 7, \Delta \approx 0.1286$
    * Worst-case error $\leq 0.0643$
    * Variance $\approx \frac{0.90^2}{12 \cdot 7^2} \approx 0.00138$

**Takeaway:** increasing bits from 2 to 4 reduces variance by factor $\approx 49$ here (since $Q^2$ grows from $1^2$ to $7^2$).

### Numeric comparison (same block max, different $k$)

Take $\max|x| = 0.90$.

* **For $k = 2$:** $Q = 1, \Delta = 0.90$
    * Worst-case error $\leq 0.45$
    * Variance $\approx \frac{0.90^2}{12 \cdot 1^2} = 0.0675$

* **For $k = 4$:** $Q = 7, \Delta \approx 0.1286$
    * Worst-case error $\leq 0.0643$
    * Variance $\approx \frac{0.90^2}{12 \cdot 7^2} \approx 0.00138$

**Takeaway:** increasing bits from 2 to 4 reduces variance by factor $\approx 49$ here (since $Q^2$ grows from $1^2$ to $7^2$).

### Metadata and memory cost

* If you store one 32-bit float scale per block of $B$ elements, the **per-element metadata cost** is:

$$\text{bits per element for scale} = \frac{32}{B}$$

* **Example:** $B = 64 \rightarrow 0.5 \text{ bits/element}$; $B = 8 \rightarrow 4 \text{ bits/element}$. Small blocks can make metadata non-negligible. Double-quantization compresses these scales to reduce that overhead.


#### Symmetric Quantization vs non-Symmetric Quantization

![Symmetric Quantization vs non-Symmetric Quantization fig](image-4.png)

## Deal with Non‑uniform formats (e.g., NF4) & Zero‑point / asymmetric quantization

What NF4 (NormalFloat 4‑bit) aims to do ?

**NF4 is a non‑uniform**, `4‑bit quantization` format designed so the 16 representable codes match the typical normal (Gaussian) distribution of pretrained transformer weights. Instead of using uniformly spaced levels inside each block, NF4 places more codes near zero and fewer far out in the tails. That reduces average quantization error for normally distributed weights while still using only 4 bits per weight.

#### NF4 algorithm:

### Gaussian Quantile Quantization (NF4 Variant)

This method builds a custom 4-bit codebook tailored to each block of weights, assuming the weights follow a normal distribution. Instead of uniform spacing, the codebook positions are spaced non-uniformly to give more precision to values close to the mean $\mu$, where most weights reside.

#### Step 1 — Estimate block statistics

For a block of weights $B = \{x_j\}$, compute the sample mean $\mu$ and sample standard deviation $\sigma$:

$$\mu = \frac{1}{|B|}\sum_{j \in B} x_j, \qquad \sigma = \sqrt{\frac{1}{|B|}\sum_{j \in B} (x_j - \mu)^2}$$

#### Step 2 — Compute standard normal code positions

To generate 16 points that partition a standard normal distribution $\mathcal{N}(0, 1)$ into equal-probability intervals, use the midpoints of 16 uniform probability bins:

$$p_i = \frac{i - 0.5}{16} \quad \text{for } i = 1, 2, \dots, 16$$

Then find the $z$-scores using the standard normal inverse Cumulative Distribution Function (percent point function):

$$z_i = \Phi^{-1}(p_i)$$

## Alternatye of $z_{i}$:

The formula

$$q_i = \frac{1}{2}\left(Q_X\left(\frac{i}{2^k + 1}\right) + Q_X\left(\frac{i + 1}{2^k + 1}\right)\right)$$

builds **codebook centroids** for a $k$-bit quantizer by taking midpoints of adjacent quantiles of a standard normal distribution $X \sim \mathcal{N}(0,1)$.

* **$Q_X(p)$** is the quantile (inverse CDF) of the standard normal at probability $p$.
* For **$k$ bits** there are $2^k$ code indices; the formula uses the $2^k + 1$ denominator to place $2^k$ intervals inside $(0, 1)$.
* Each **$q_i$** is the midpoint between two adjacent normal quantiles, so the set $\{q_i\}$ partitions the real line into Voronoi cells whose boundaries are the quantiles.
* **Intuition:** these $q_i$ are the optimal representative values (in a quantile sense) for a Gaussian distribution — many representatives near zero, fewer in the tails.

#### Step 3 — Scale and shift to the block

Transform the standard normal quantiles to match the block's specific distribution:

$$c_i = \mu + \sigma \cdot z_i$$

Because $z_i$ is dense near 0 and sparse in the tails, $c_i$ matches the distribution profile of neural network weights perfectly.

Or if you have used $Q_{i}$
This maps the standard-normal centroids to the block's distribution.

$$c_i = \mu + \sigma \cdot q_i$$

### Steps 4 and 5 — Assignment and Storage

Once the codebook values $c_i$ are determined for the block, each original floating-point weight is mapped to its closest representative and stored efficiently.

#### Step 4 — Quantize each weight to nearest code

Map each weight $x_j$ in the block to the specific codebook entry $c_i$ that minimizes the absolute distance:

$$x_j^q = \arg\min_{c_i} |x_j - c_i|$$

#### Step 5 — Store the 4-bit index and block metadata

For each weight, save only the 4-bit integer index corresponding to $x_j^q$. Along with these indices, store the block metadata $(\mu, \sigma)$ (or a compressed form of them, such as via double-quantization).

This produces a **per-block, Gaussian-matched 16-entry codebook** that highly accurately approximates the distribution of weights in that block.

### Expected Mean Squared Error (MSE) Optimization

The choice of codebook values $\{c_i\}$ dictates how much precision is lost during quantization. When weights follow a continuous distribution with density $f(x)$, we optimize for the **expected squared error** rather than a uniform worst-case bound.

#### Step 1 — Expected MSE over Voronoi regions

Let $\hat{x}$ be the quantized value of $x$. The continuous expectation is split into a sum of integrals over disjoint partitions called **Voronoi regions** $R_i$:

$$E\left[(x - \hat{x})^2\right] = \sum_{i} \int_{R_i} (x - c_i)^2 f(x) \, dx$$

where each region $R_i$ maps any value $x$ to its nearest codebook representative $c_i$:

$$R_i = \left\{ x \;\middle|\; |x - c_i| \leq |x - c_j| \; \forall j \right\}$$

#### Step 2 — Impact of a Gaussian distribution

When $f(x)$ is a Gaussian distribution $\mathcal{N}(\mu, \sigma^2)$:
* **High-density regions (near $\mu$):** $f(x)$ is large. To keep the total integral small, the width of $R_i$ must be narrow. This places more codes close together near the center.
* **Low-density regions (the tails):** $f(x)$ is very small. The regions $R_i$ can afford to be wider because the low probability weight suppresses the overall error contribution.

#### Step 3 — Quantile vs. Uniform distribution alignment

* **Quantile / Non-uniform quantization:** Minimizes **expected error** under a Gaussian by adapting region widths directly to the density function $f(x)$.
* **Uniform quantization:** Minimizes the **worst-case absolute error** over a bounded interval, making it suboptimal for bell-shaped weight distributions since it wastes code positions on empty tail regions.


### Worst-Case Error Bounds for Non-Uniform Quantization (NF4)

When analyzing quantization limits, the metric for maximum degradation changes significantly from uniform grids to non-uniform, distribution-aware codebooks like **NF4 (NormalFloat 4)**.

#### Step 1 — Contrast with uniform bounds

For uniform symmetric quantization, code positions are evenly spaced across the entire dynamic range. The worst-case absolute error per element depends exclusively on the maximum block value and the maximum quantization integer $Q$:

$$|x - \hat{x}|_{\text{uniform}} \leq \frac{\max|x|}{2Q}$$

#### Step 2 — Deriving the non-uniform bound

For non-uniform codebooks, the error is no longer uniform across the distribution. Instead, any real value $x$ bounded by the minimum and maximum codebook values falls between two sorted, adjacent codes $c_i$ and $c_{i+1}$. 

The worst-case absolute error within that specific interval is exactly half the distance between those two points. Over the entire codebook, the **worst-case absolute error bound** is defined by half the largest adjacent codebook gap:

$$|x - \hat{x}|_{\text{NF4}} \leq \frac{1}{2} \max_{i} (c_{i+1} - c_i)$$

#### Step 3 — Mapping to NF4 constants

In a standard NF4 codebook scaled to a specific block:
* The largest adjacent gaps occur at the far edges (**extreme codes** in the tails), such as between $c_2 - c_1$ or $c_{16} - c_{15}$.
* For a normalized block, this maximum adjacent gap is approximately **$0.108$**.
* Dividing by 2 yields a conservative, strict upper bound for the maximum absolute quantization error:

$$|x - \hat{x}| \leq 0.054$$

Because most weights lie near the mean where gaps are tight, the actual observed maximum error (e.g., $0.026$) typically lands well below this theoretical threshold.


#### Example 1:

### Worked Numerical Example with Gaussian Quantiles

Using the block of 8 weights provided in the image:

$$B = [0.30, -0.12, 0.05, -0.25, 0.18, -0.05, 0.40, -0.20]$$

#### 1. Compute block mean and standard deviation
As calculated in the image (rounded):

$$\mu \approx 0.03, \qquad \sigma \approx 0.20$$

#### 2. Obtain standard normal quantiles ($z$-scores)
Using $p = \frac{i - 0.5}{16}$, the lower half of the 16 symmetric values yield the following approximate $z$-scores:

| $i$ | $p$ | $z = \Phi^{-1}(p)$ |
| :--- | :--- | :--- |
| **1** | $0.03125$ | $-1.86$ |
| **2** | $0.09375$ | $-1.32$ |
| **3** | $0.15625$ | $-1.02$ |
| **4** | $0.21875$ | $-0.77$ |
| **5** | $0.28125$ | $-0.58$ |
| **6** | $0.34375$ | $-0.40$ |
| **7** | $0.40625$ | $-0.24$ |

*(The remaining upper half values from $i=9$ to $16$ are positive and mirror these exact magnitudes symmetrically around zero).*

### Step B and Step C — Codebook Generation and Assignment

Using the statistics **$\mu \approx 0.03$** and **$\sigma \approx 0.20$** from the previous step, we can scale the standard normal quantiles to form the actual localized codebook and quantize weights against it.

#### Step B — Form codebook entries $c_i = \mu + \sigma z_i$

The representative codebook entries are calculated symmetrically around the mean:

* **$c_{\text{neg large}}$**: $0.03 + 0.20 \cdot (-1.86) \approx -0.339$
* **$c_{\text{neg med}}$**: $0.03 + 0.20 \cdot (-1.02) \approx -0.174$
* **$c_{\text{neg small}}$**: $0.03 + 0.20 \cdot (-0.24) \approx -0.018$
* **$c_0$ (center)**: $\approx 0.03$
* **$c_{\text{pos small}}$**: $0.03 + 0.20 \cdot 0.24 \approx 0.078$
* **$c_{\text{pos med}}$**: $0.03 + 0.20 \cdot 1.02 \approx 0.234$
* **$c_{\text{pos large}}$**: $0.03 + 0.20 \cdot 1.86 \approx 0.402$

#### Step C — Quantize two example weights

Each real-valued weight is mapped to its closest target code value from the generated codebook:

* **For $x = 0.30$**: The two closest positive options are $c_{\text{pos med}} \approx 0.234$ and $c_{\text{pos large}} \approx 0.402$. 
  * Nearest code: **$0.234$**
  * Quantization error: $0.30 - 0.234 = \mathbf{0.066}$

* **For $x = -0.25$**: The two closest negative options are $c_{\text{neg med}} \approx -0.174$ and $c_{\text{neg large}} \approx -0.339$. 
  * Nearest code: **$-0.174$**
  * Quantization error: $-0.25 - (-0.174) = \mathbf{-0.076}$

### Comparison with Uniform 4-bit Symmetric Quantization (Same Block)

To evaluate the efficiency of the Gaussian quantile approach, we compare it against a standard uniform symmetric 4-bit scheme on the exact same block.

#### 1. Setup uniform quantization parameters
Uniform symmetric 4-bit quantization maps values to integers between $[-7, 7]$, meaning $Q = 7$. For this block, the maximum absolute value is $\max|x| = 0.40$:

* **Scale ($s$):** 

$$s = \frac{Q}{\max|x|} = \frac{7}{0.40} = 17.5$$

* **Step size ($\Delta$):** 

$$\Delta = \frac{1}{s} \approx 0.05714$$

The quantized grid levels are uniform multiples of $\Delta$ centered around zero:

$$\dots, -2\Delta, -\Delta, 0, \Delta, 2\Delta, \dots$$

#### 2. Quantize the same two example weights

* **For $x = 0.30$:** The nearest uniform dequantized level is $5\Delta \approx 0.2857$.
  * **Quantization error:** $0.30 - 0.2857 = \mathbf{0.0143}$

* **For $x = -0.25$:** The nearest uniform level is $-4\Delta \approx -0.2286$.
  * **Quantization error:** $-0.25 - (-0.2286) = \mathbf{-0.0214}$

#### Observation and Takeaway

In this isolated, small block example, the uniform scheme actually yields smaller individual errors for these two values. This happens because the small block maximum ($\max|x| = 0.40$) shrinks the step size $\Delta$ significantly, allowing the uniform grid lines to fortuitously align closely with these specific numbers. 

However, this is **not the typical case** across massive neural network layers. For large, normally distributed weights spread across millions of parameters, the non-uniform Gaussian codebook (like NF4) consistently produces a lower global Mean Squared Error (MSE) by focusing representation density where the bulk of the weights actually reside.


## QLoRA with LoRA

For further processing we need to understand formula and what it means

![High‑level LoRA_a and LoRA_b](image-5.png)

The image is a schematic of LoRA (Low‑Rank Adapters) showing that for each target module in a transformer you attach two small adapter matrices (often called LoRA_a and LoRA_b). At runtime the adapter output is combined with the module’s original output so the frozen base model’s behavior is adjusted by a small, trainable low‑rank correction. The diagram emphasizes modularity (many target modules, each with its own pair of LoRA matrices) and the fact that only the adapters are trained while the large base weights remain frozen.

The equation

$$Y = XW + s XL_1L_2$$

* $XW$ is the original linear projection performed by the target module (e.g., a query, key, value, or feed-forward projection).
* $L_1$ and $L_2$ are the two LoRA factors (often called LoRA A and LoRA B). Their product $L_1L_2$ is a low-rank matrix that approximates a full-rank update.
* $s$ (sometimes written as $\alpha/r$ in papers) is a scalar **scaling factor** that controls the magnitude of the adapter contribution.
* The adapter term is written as $s XL_1L_2$ because it is computed efficiently as $(XL_1)L_2$ — first project $X$ into a small $r$-dimensional space, then project back to the original output dimension.

To understand shape of above equations first to understand product of two matrix

The output dimensions of multiplying an **$(m \times n)$ matrix** by an **$(n \times l)$ matrix** are exactly **$m \times l$**.

#### Matrix Dimensions

* **First Matrix:** $m$ rows, $n$ columns.
* **Second Matrix:** $n$ rows, $l$ columns.
* **Condition:** Inner dimensions ($n$) must match.
* **Result Matrix:** Outer dimensions ($m \times l$).

#### Matrix dimensions and why they are chosen in QLoRA

#### Matrix dimensions and why they are chosen

To make the matrix multiplications valid and efficient the shapes are chosen as:

* $X$: $(b, in)$ — batch size $b$ by input dimension $in$.
* $W$: $(in, out)$ — original weight matrix; $XW \rightarrow (b, out)$.
* $L_1$: $(in, r)$ — projects input into a low-rank space of size $r$.
* $L_2$: $(r, out)$ — projects back to the output dimension.
* $s$: scalar.

#### Computation order and intermediate shapes:

1. $XL_1$ has shape $(b, r)$.
2. $(XL_1)L_2$ has shape $(b, out)$.
3. Add to $XW$ to get $Y$ of shape $(b, out)$.

**Why this is efficient:** $r$ is chosen small (e.g., 4, 8, 16), so the intermediate $(b, r)$ is cheap. Parameter count for adapters is $in \cdot r + r \cdot out$, which is tiny compared to $in \cdot out$.



### Numeric example (concrete)

If $in = 768$, $out = 3072$, $b = 4$, and $r = 8$:

* $W$ has $768 \times 3072 = 2,359,296$ params.
* **LoRA params:** $768 \times 8 + 8 \times 3072 = 6,144 + 24,576 = 30,720$ params.
* Compute $XL_1 \rightarrow (4, 8)$, then $(XL_1)L_2 \rightarrow (4, 3072)$, then add scaled adapter to $XW$.

This shows the adapter uses **~77× fewer parameters** for that projection.

#### LoRA interface with BERT

In a typical setup you add LoRA to the attention projections and the feed‑forward projections, which for BERT‑base (12 layers) gives 6 target projections per layer → 72 LoRA pairs total. You can apply fewer or more depending on your design.

#### Where LoRA is usually applied in BERT (per transformer layer)

* **Attention projections:** Query ($Q$), Key ($K$), Value ($V$), and Output ($O$) — **4 projections**.
* **Feed-forward network (FFN):** the first (up) projection and the second (down) projection — **2 projections**.
* **Total per layer (typical):** $4 + 2 = 6$ target modules.

So for a model with $L$ layers, typical total LoRA pairs $= 6L$.

#### Concrete counts

* **BERT-base:** $L = 12 \rightarrow 6 \times 12 = 72$ LoRA pairs.
* **BERT-large:** $L = 24 \rightarrow 6 \times 24 = 144$ LoRA pairs.
* **If you only adapt attention Q and V:** $2$ per layer $\rightarrow 2L$ pairs (e.g., $24$ for BERT-base).

Each "pair" means two small matrices $L_1$ and $L_2$ whose product forms the low-rank update for that projection.

### Parameter math (how many extra params per LoRA pair)

If a projection has input dim $d_{\text{in}}$ and output dim $d_{\text{out}}$, and you choose LoRA rank $r$, the adapter parameter count is

$$\text{params per pair} = d_{\text{in}} \cdot r + r \cdot d_{\text{out}}$$

Common case in BERT projections: $d_{\text{in}} = d_{\text{out}} = H$ (hidden size). For BERT-base $H = 768$.

**Example with $r = 8$ on a projection with $H = 768$:**

$$\text{params per pair} = 768 \cdot 8 + 8 \cdot 768 = 2 \cdot (768 \cdot 8) = 12,288$$

If you add LoRA to the 6 projections in one layer: $6 \times 12,288 = 73,728$ params per layer. For 12 layers: $73,728 \times 12 = 884,736$ extra parameters total (under 1M).


#### How many LoRA pairs in BERT


| Model | Layers | Projections per layer | LoRA pairs total |
| :--- | :--- | :--- | :--- |
| BERT-base | 12 | 6 (Q,K,V,O,FFN up, FFN down) | **72** |
| BERT-large | 24 | 6 | **144** |
| Minimal setup | $L$ | 2 (Q,V only) | **2L** |


Each “pair” = two matrices $L_1$,$L_2$. You may adapt fewer projections to save parameters.

### Why two matrices instead of one

* **Dimensionality matching:** $W$ is $(\text{in}, \text{out})$. Two small matrices $(\text{in}, r)$ and $(r, \text{out})$ multiply to that shape while keeping parameter count low.
* **Compute efficiency:** compute $(XL_1)$ first (shape $(b, r)$), then multiply by $L_2$ to get $(b, \text{out})$. This uses the small intermediate dimension $r$ and is cheap.
* **Parameter efficiency:** $\text{params per pair} = \text{in} \cdot r + r \cdot \text{out}$, much smaller than $\text{in} \cdot \text{out}$ when $r \ll \min(\text{in}, \text{out})$.
### Scaling and initialization

* **Scaling factor $s$** (often $\alpha/r$) controls the adapter magnitude so the adapter does not overwhelm pretrained weights.
* **Initialization:** common practice initializes $L_2$ to zeros or small values and $L_1$ randomly so the adapter starts near zero and the pretrained model behavior is preserved.


#### Equations $Y = XW + s XL_1L_2$ in terms of QLoRA 

### The two equations (restated)

$$Y^{\text{BF16}} = X^{\text{BF16}} \text{doubleDequant}(c_1^{\text{FP32}}, c_2^{k\text{-bit}}, W^{\text{NF4}}) + X^{\text{BF16}}L_1^{\text{BF16}}L_2^{\text{BF16}} \quad -- (5)$$

$$\text{doubleDequant}(c_1^{\text{FP32}}, c_2^{k\text{-bit}}, W^{k\text{-bit}}) = \text{dequant}(\text{dequant}(c_1^{\text{FP32}}, c_2^{k\text{-bit}}), W^{\text{4bit}}) = W^{\text{BF16}} \quad (6)$$

Equations (5) and (6) describe one forward pass of a single linear layer under QLoRA: the frozen base weight matrix is stored in a highly compressed 4‑bit format and is dequantized twice into the BF16 computation type for the forward/backward pass, while a small LoRA adapter (two BF16 matrices) is computed and added. Only the LoRA parameters are updated during training.

Where

* $X^{\text{BF16}}$ — input activations in BF16 (computation precision).
* $W^{\text{NF4}}$ — base layer weights stored in **NF4** (4-bit NormalFloat) per block. This is the *storage representation*.
* $c_2^{k\text{-bit}}$ — per-block quantization metadata (scales/offsets) themselves quantized to a small format (the paper uses FP8 for these).
* $c_1^{\text{FP32}}$ — higher-precision metadata (or global constants) stored in FP32 used together with $c_2$ to reconstruct the true per-block scale.
* $L_1^{\text{BF16}}, L_2^{\text{BF16}}$ — LoRA adapter matrices (both in BF16), with shapes $(\text{in}, r)$ and $(r, \text{out})$.
* $\text{doubleDequant}(\cdot)$ — the two-step dequantization that reconstructs a BF16 version of the stored 4-bit weights.
* **Result:** $W^{\text{BF16}}$ is the dequantized weight used for matrix multiply with $X^{\text{BF16}}$.

### Why "double" dequantization

**Double quantization** compresses not only the weight integers but also the per-block scale metadata to save memory. To reconstruct the usable floating weights you must:

1. **Dequantize the compressed metadata** $c_2^{k\text{-bit}}$ using the higher-precision constant $c_1^{\text{FP32}}$ to recover the per-block scale $s$.
   * Conceptually: $s = \text{dequant\_scale}(c_1, c_2)$.

2. **Dequantize the 4-bit integer weight values** using that recovered scale:
   * Conceptually: $W^{\text{BF16}} = \text{dequant\_weights}(W^{\text{4bit}}, s)$.

This two-step process is what equation (6) expresses compactly.

### Concrete numeric sketch of the two steps

* Suppose a block stores integer codes $w^{\text{int}} \in \{-7, \dots, 7\}$ (NF4 style) and the block scale is stored in two compressed pieces $c_2$ (FP8) and $c_1$ (FP32 helper).
* **Step 1 (metadata dequant):** recover scale $s$ from $(c_1, c_2)$:

  $$s = \text{dequant\_scale}(c_1^{\text{FP32}}, c_2^{\text{FP8}}) \quad \text{(result in FP32/BF16)}.$$

* **Step 2 (weights dequant):** recover BF16 weights:

  $$w^{\text{BF16}} = \frac{w^{\text{int}}}{Q} \cdot s,$$

  where $Q$ is the integer range constant (e.g., $Q = 7$ for signed 4-bit symmetric).
* Use $W^{\text{BF16}}$ in the matrix multiply $X^{\text{BF16}}W^{\text{BF16}}$.

#### Full forward pass interpretation (equation 5)

* **Base contribution:** $X^{\text{BF16}}W^{\text{BF16}}$ — the frozen model's linear projection, reconstructed on the fly from compressed storage.
* **Adapter contribution:** $X^{\text{BF16}}L_1^{\text{BF16}}L_2^{\text{BF16}}$ — the LoRA low-rank correction computed in BF16.
* **Sum:** the layer output in BF16 is the sum of both terms.

#### Backward pass and gradients

* **We do not compute $\partial E/\partial W$** because $W$ is frozen (no optimizer state for base weights).
* **We do compute gradients for LoRA:** $\partial E/\partial L_1$ and $\partial E/\partial L_2$. Those gradients require the forward activations and the dequantized $W^{\text{BF16}}$ because the backward pass computes derivatives through the same forward computation graph. Concretely:
  * Compute $\delta = \partial L/\partial Y$.
  * Adapter gradients use $X$ and $\delta$: e.g., $\partial L/\partial L_2 = (XL_1)^T\delta$, $\partial L/\partial L_1 = X^T(\delta L_2^T)$.
* **Why dequantize to BF16:** the forward/backward math uses BF16 for numerical stability and speed on modern accelerators; dequantizing to BF16 gives accurate enough activations/gradients while keeping storage tiny.


### Selections of Blocksize and data-type choices

* **$W$ blocksize $= 64$** (smaller blocks $\rightarrow$ better local fit $\rightarrow$ lower quantization error).
* **$c_2$ blocksize $= 256$** (coarser blocks for metadata to save memory).
* **$W$ stored in NF4 (4-bit);** $c_2$ stored in FP8; $c_1$ in FP32; computation in BF16.
* These choices balance **memory**, **compute**, and **accuracy**.
